In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import os

In [7]:
data_dir = "./outputs-old1"
file_name = "llm_ratings.csv"

data = pd.read_csv(os.path.join(data_dir, file_name))

In [8]:
data.head()

,model_name,provider,model_id,prompt_name,product_id,dimension,llm_score,reason
0,claude_46_opus,anthropic,claude-opus-4-6,base_cot,1,cognitive,6.0,The SecuLife Guardian SOS Wristband is specifi...
1,claude_46_opus,anthropic,claude-opus-4-6,base_cot,1,physical,6.0,The SecuLife Guardian SOS Wristband is specifi...
2,claude_46_opus,anthropic,claude-opus-4-6,base_cot,2,cognitive,6.0,The Lively Mobile2 is specifically designed fo...
3,claude_46_opus,anthropic,claude-opus-4-6,base_cot,2,physical,6.0,The Lively Mobile2 is specifically designed fo...
4,claude_46_opus,anthropic,claude-opus-4-6,base_cot,3,cognitive,4.0,The NestHao Bed Sensor Alarm involves moderate...


In [46]:
base_mask = (data['prompt_name'] == 'base_cot') & (data['model_name'] != 'gemini_31_pro')
up_mask = (data['prompt_name'] == 'up_cot') & (data['model_name'] != 'gemini_31_pro')

In [58]:
base = data[base_mask][['model_name', 'product_id', 'dimension','llm_score']]
base_physical = base[base['dimension'] == 'physical'][['model_name', 'product_id', 'llm_score']]
base_cognitive = base[base['dimension'] == 'cognitive'][['model_name', 'product_id', 'llm_score']]

In [60]:
up = data[up_mask][['model_name', 'product_id', 'dimension', 'llm_score']]
up_physical = up[up['dimension'] == 'physical'][['model_name', 'product_id', 'llm_score']]
up_cognitive = up[up['dimension'] == 'cognitive'][['model_name', 'product_id', 'llm_score']]

In [62]:
# Align on product_id; validate catches duplicates
paired = base.merge(
    up,
    on=['model_name', 'product_id'],
    how="inner",
    suffixes=("_base", "_up"),
    validate="one_to_one"
)

print("paired rows:", len(paired))


MergeError: Merge keys are not unique in either left or right dataset; not a one-to-one merge.
Duplicates in left:
     model_name  product_id
claude_46_opus           1
claude_46_opus           2
claude_46_opus           3
claude_46_opus           4
claude_46_opus           5 ...
Duplicates in right:
     model_name  product_id
claude_46_opus           1
claude_46_opus           2
claude_46_opus           3
claude_46_opus           4
claude_46_opus           5 ...

In [96]:
def check_match(base, up, keys: list[str]):
    base_keys = base[keys]
    up_keys = up[keys]

    base_only = base_keys.merge(up_keys, on=keys, how="left", indicator=True)
    base_only = base_only[base_only["_merge"] == "left_only"]

    up_only = up_keys.merge(base_keys, on=keys, how="left", indicator=True)
    up_only = up_only[up_only["_merge"] == "left_only"]

    return base_only, up_only

def get_matching_keys(base, up, keys: list[str]):
    base_keys = base[keys]
    up_keys = up[keys]

    matching_keys = base_keys.merge(up_keys, on=keys, how="inner", indicator=True)
    matching_keys = matching_keys.drop(columns='_merge')

    return matching_keys


In [97]:
base_phy = base[base['dimension'] == 'physical']
base_cog = base[base['dimension'] == 'cognitive']
up_phy = up[up['dimension'] == 'physical']
up_cog = up[up['dimension'] == 'cognitive']

key = ['model_name', 'product_id']
base_only, up_only = check_match(base_phy, up_phy, key)
matching_phy_keys = get_matching_keys(base_phy, up_phy, key)

print("keys in base_phy not in up_phy:", len(base_only))
print("keys in up_phy not in base_phy:", len(up_only))
# Now safe for paired test
# t_stat, p_val = stats.ttest_rel(paired["llm_score_base"], paired["llm_score_up"])

keys in base_phy not in up_phy: 1
keys in up_phy not in base_phy: 0


In [101]:
base_phy_unique = base_phy.merge(matching_phy_keys, on=["model_name", "product_id"], how="inner")
up_phy_unique = up_phy.merge(matching_phy_keys, on=["model_name", "product_id"], how="inner")
print(len(base_phy_unique[base_phy_unique['model_name']=='openai_gpt4']) == len(up_phy_unique[up_phy_unique['model_name']=='openai_gpt4']))

True


In [110]:
gpt_result = stats.ttest_rel(base_phy_unique[base_phy_unique['model_name']=='openai_gpt4']['llm_score'], up_phy_unique[up_phy_unique['model_name']=='openai_gpt4']['llm_score'])

In [111]:
gpt_result

TtestResult(statistic=np.float64(13.690091565089329), pvalue=np.float64(1.1847985215189553e-17), df=np.int64(45))

In [112]:
def paired_ttest_by_model(base_df, up_df, model_name, keys=['model_name', 'product_id']):
    base_model = base_df[base_df['model_name'] == model_name][keys + ['llm_score']].copy()
    up_model = up_df[up_df['model_name'] == model_name][keys + ['llm_score']].copy()

    paired = base_model.merge(
        up_model,
        on=keys,
        how='inner',
        suffixes=('_base', '_up'),
        validate='one_to_one',
    )
    paired = paired.dropna(subset=['llm_score_base', 'llm_score_up'])

    if len(paired) != len(base_model) or len(paired) != len(up_model):
        print(f'{model_name}: base={len(base_model)}, up={len(up_model)}, paired={len(paired)}')

    return stats.ttest_rel(paired['llm_score_base'], paired['llm_score_up']), paired

paired_results = {}
for dim_name, base_df, up_df in [('physical', base_phy, up_phy), ('cognitive', base_cog, up_cog)]:
    for model_name in ['claude_46_opus', 'openai_gpt4']:
        result, paired = paired_ttest_by_model(base_df, up_df, model_name)
        paired_results[(dim_name, model_name)] = result
        print(dim_name, model_name, result)
        print('paired rows:', len(paired))

physical claude_46_opus TtestResult(statistic=np.float64(7.910670694789247), pvalue=np.float64(4.0062571723324443e-10), df=np.int64(46))
paired rows: 47
openai_gpt4: base=47, up=46, paired=46
physical openai_gpt4 TtestResult(statistic=np.float64(13.690091565089329), pvalue=np.float64(1.1847985215189553e-17), df=np.int64(45))
paired rows: 46
claude_46_opus: base=43, up=44, paired=40
cognitive claude_46_opus TtestResult(statistic=np.float64(9.357765974865162), pvalue=np.float64(1.6198295289777496e-11), df=np.int64(39))
paired rows: 40
openai_gpt4: base=47, up=41, paired=41
cognitive openai_gpt4 TtestResult(statistic=np.float64(18.244401744182134), pvalue=np.float64(5.394235251992508e-21), df=np.int64(40))
paired rows: 41
